<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/26_phase3_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 26 — Faz 3 değerlendirmesi: fine-tune sonuçları (EXP-030; CPU)

**Ne yapıyor?** Notebook 25'in Drive'daki fold tahminlerini (`preds.csv`), aynı gün-gruplu split'lerdeki dondurulmuş eşlerle (EXP-016DG) karşılaştırır. Tanımlar `docs/PHASE3_DESIGN.md` 5.2'de sonuçlardan önce yazıldı:
- **E1:** fold AUC ort ± SD;
- **T1:** bağlamın (saat, tarih, yaş) ötesinde bilgi;
- **C3:** fine-tune dondurulmuştan daha fazla bilgi taşıyor mu (keşifsel);
- **C2:** bağlam kullanımı;
- **RQ2:** ön-eğitim (cnn10 − sıfırdan).
- **O1** (betimsel): aşırı uyum / kalibrasyon.

Yeniden eğitim yok. GPU gerekmez: *Runtime → CPU*. Süre ~10–30 dk [tahmin]. Rapor yalnız agrega; katılımcı düzeyi tablo Drive'a yazılır.

## 0. Oturum kurulumu

In [ ]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bu notebook CPU üzerinde çalışır; sorun değil.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
BACKBONES   = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

In [ ]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

In [ ]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import numpy as np, pandas as pd, sklearn, statsmodels
print('numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__, '| statsmodels', statsmodels.__version__)

In [ ]:
# Kapı: kod ve kararlar main'de; 4 kolun 15 fold'u Drive'da bitmiş; dondurulmuş eşler ve D-035 seçimi var
for p in ['docs/PHASE3_DESIGN.md', 'scripts/evaluate_phase3.py', 'reports/phase3/G1_frozen_daygrouped.json']:
    if git('ls-files', '--error-unmatch', p, check=False).returncode != 0:
        raise RuntimeError(f'{p} main dalında yok: PR birleştirildi mi? Sonra 0. bölümü yeniden çalıştır.')
dec = (REPO_DIR / 'DECISIONS.md').read_text()
assert '## D-043' in dec, 'D-043 main dalında yok'
SPLITS_DAY = DERIVED / 'splits_day'
PROBE_DAY = EXPERIMENTS / 'phase3' / 'frozen_probe_day'
FT_OUT = EXPERIMENTS / 'phase3' / 'finetune'
SEL = PROBE_DAY / 'd035_selection.csv'
assert SEL.exists(), 'd035_selection.csv yok (notebook 24)'
for arm in ['cnn10', 'cnn10_scratch', 'ssl', 'cnn14']:
    n = len(list(FT_OUT.glob(f'{arm}/r*_k*/fold.json')))
    assert n == 15, f'{arm}: {n}/15 fold bitmiş → notebook 25 Bölüm 2'
print('Kapı geçti: 4 kol × 15 fold.')

## 1. Değerlendirme (EXP-030)
C3 bootstrap'ı (her örneklemde 4 Firth fit'i) en uzun adımdır; bütün çekirdekler kullanılır.

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/evaluate_phase3.py finetune --ft-root {FT_OUT} --probe-dir {PROBE_DAY} --selection {SEL} '
   f'--splits-dir {SPLITS_DAY} --context {DERIVED}/participant_context.csv --report-dir {REPO_DIR}/reports/phase3 '
   f'--out-dir {EXPERIMENTS}/phase3')

## 2. Raporu GitHub'a gönder (yeni dal)

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase3-eval', ['reports/phase3/EXP-030_finetune.md', 'reports/phase3/EXP-030_finetune.json'], 'Faz 3 EXP-030 fine-tune değerlendirmesi (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')